In [ ]:
import sys
import os
sys.path.insert(0, "..")  # Adds the parent directory (project root) to path

from src.scanner import scan_library
from src.store import init_duckdb_connection, create_library_table, insert_file_records
from src.reports import human_readable_size


In [ ]:
duckdb_conn = init_duckdb_connection(os.path.join("..", "data", "library.duckdb"))
library_table = create_library_table(duckdb_conn)
library_data = scan_library()
insert_file_records(duckdb_conn, library_data)

In [ ]:
df = duckdb_conn.sql("SELECT * FROM library")
df.show()

In [ ]:
df = duckdb_conn.sql("""
    SELECT type, SUM(size) AS Total_Size, COUNT(*) AS File_Count 
    FROM library 
    GROUP BY type 
    ORDER BY Total_Size DESC
    """).df()

df['Total_Size'] = df['Total_Size'].apply(human_readable_size)
display(df)